# Kernel DMD

`dmd.kernelDMD` with a linear kernel, which should recover exact DMD and so checks the method, and a degree-2 polynomial kernel as in `CMECS-kDMD`.

Data: GLORYS04 SSH (364 days) and surface U and V in the Gulf Stream (731 days, 30-45N, 75-45W). Animations show the anomaly beside its reconstruction.

In [ ]:
import numpy as np
from showcase import *
from modredkit import dmd, pod

RANK = 50

## SSH

In [ ]:
A, area, wet = load_ssh()
print("SSH anomalies", A.shape)
S2 = A.T @ A
exact = dmd.exactDMD(A[:, :-1], A[:, 1:], RANK)[1]
mds, ct, eig, *_ = dmd.kernelDMD(S2, A[:, :-1].T, DT, RANK)
b = dmd.DMD_amplitudes_optimal(mds, eig, A)
print(f"linear: relative error {relative_error(rebuild(mds, b, eig, A.shape[1]), A):.3f}; largest gap to exact DMD eigenvalues {np.max([np.min(np.abs(eig-e)) for e in exact]):.1e}")
show_spectrum(ct, b, "SSH: linear kDMD spectrum")
S2p = (1 + S2/np.mean(np.diag(S2)))**2
mds, ct, eig, *_ = dmd.kernelDMD(S2p, A[:, :-1].T, DT, RANK)
b = dmd.DMD_amplitudes_optimal(mds, eig, A)
print(f"polynomial: relative error {relative_error(rebuild(mds, b, eig, A.shape[1]), A):.3f}; {np.sum(np.abs(eig) > 1)} of {eig.size} modes grow")
show_spectrum(ct, b, "SSH: polynomial kDMD spectrum (degree 2)")

In [ ]:
rec = rebuild(mds, b, eig, A.shape[1])
animate([A, rec], ["SSH anomaly", "Polynomial kDMD, 50 modes"], wet, units="m")

## Gulf Stream U and V

In [ ]:
U, V, wet_u, wet_v = load_gulfstream()
Z = np.vstack((U, V))
nu = U.shape[0]
print("U", U.shape, "V", V.shape)
S2 = Z.T @ Z
exact = dmd.exactDMD(Z[:, :-1], Z[:, 1:], RANK)[1]
mds, ct, eig, *_ = dmd.kernelDMD(S2, Z[:, :-1].T, DT, RANK)
b = dmd.DMD_amplitudes_optimal(mds, eig, Z)
print(f"linear: relative error {relative_error(rebuild(mds, b, eig, Z.shape[1]), Z):.3f}; largest gap to exact DMD eigenvalues {np.max([np.min(np.abs(eig-e)) for e in exact]):.1e}")
show_spectrum(ct, b, "Gulf Stream: linear kDMD spectrum")
S2p = (1 + S2/np.mean(np.diag(S2)))**2
mds, ct, eig, *_ = dmd.kernelDMD(S2p, Z[:, :-1].T, DT, RANK)
b = dmd.DMD_amplitudes_optimal(mds, eig, Z)
print(f"polynomial: relative error {relative_error(rebuild(mds, b, eig, Z.shape[1]), Z):.3f}; {np.sum(np.abs(eig) > 1)} of {eig.size} modes grow")
show_spectrum(ct, b, "Gulf Stream: polynomial kDMD spectrum (degree 2)")

In [ ]:
rec = rebuild(mds, b, eig, Z.shape[1])
animate([U, rec[:nu]], ["U anomaly", "Polynomial kDMD, 50 modes"], wet_u, step=12, units="m/s")